In [46]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import Text, create_engine
from sqlalchemy.dialects.mysql import LONGTEXT
import ast

In [60]:
DATA_DIR = Path.cwd().parent / "data"
DATA_PATH = DATA_DIR / "supreme_court.parquet"
DATA_CSV = DATA_DIR / "supreme_court.csv"

In [14]:
df = pd.read_csv(DATA_CSV)
df.head()

,Unnamed: 0,id,klasifikasi,sub_klasifikasi,paragraphs
0,0,568053a63dd9072c89d42c07709e529f,pidana-khusus,pidana-khusus,"{'tag': array(['kepala_putusan', 'riwayat_pena..."
1,1,a310b11e400de5bb7bf59aec7c9c2c59,pidana-khusus,pidana-khusus,"{'tag': array(['kepala_putusan', 'identitas', ..."
2,2,fa9e0d9079091f5ef70fef8c63f81876,pidana-umum,pidana-umum,"{'tag': array(['kepala_putusan', 'identitas', ..."
3,3,a5424228a2a06a7a659c67189582d3b5,pidana-khusus,pidana-khusus,"{'tag': array(['kepala_putusan', 'identitas', ..."
4,4,41dc9c80177fde94e63ed0d3cb6f5461,pidana-khusus,pidana-khusus,"{'tag': array(['kepala_putusan', 'identitas', ..."


In [21]:
df.columns

Index(['Unnamed: 0', 'id', 'klasifikasi', 'sub_klasifikasi', 'paragraphs'],
      dtype='str')

In [ ]:
# load data into supreme_court table
engine = create_engine(
    "mysql+pymysql://root:@127.0.0.1:3306/taxonomy_semantic?charset=utf8mb4"
)

df.to_sql(
  name="supreme_court",
  con=engine,
  if_exists="append",
  index=False,
  chunksize=500,
  dtype={
      "id": Text(),
      "klasifikasi": Text(),
      "sub_klasifikasi": Text(),
      "paragraphs": LONGTEXT(),
  },
)

22630

In [ ]:
# read data csv
def array(data, dtype=None):
  return np.array(data, dtype=dtype)

raw_str = df["paragraphs"].iloc[1]
parsed = eval(
    raw_str, {"__builtins__": {}}, {"array": array, "object": object}
)

print(type(parsed), list(parsed.keys()))
print(parsed["tag"])

<class 'dict'> ['tag', 'value']
['kepala_putusan' 'identitas' 'riwayat_penahanan' 'riwayat_perkara'
 'riwayat_tuntutan' 'riwayat_dakwaan' 'fakta' 'fakta_hukum'
 'pertimbangan_hukum' 'amar_putusan' 'penutup']


In [ ]:
# load data into documents and paragraphs tables
engine = create_engine("mysql+pymysql://root:@127.0.0.1:3306/taxonomy_semantic?charset=utf8mb4")

df = pd.read_parquet(DATA_PATH)
print(df.shape, df.columns.tolist())

docs = df[["id", "klasifikasi", "sub_klasifikasi"]]
docs.to_sql("documents", engine, if_exists="append", index=False, chunksize=1000)

rows = []
for doc_id, p in zip(df["id"], df["paragraphs"]):
    for pos, (t, v) in enumerate(zip(p["tag"], p["value"])):
        rows.append((doc_id, pos, t, v))

paras = pd.DataFrame(rows, columns=["doc_id", "position", "tag", "value"])
print(len(paras))
paras.to_sql("paragraphs", engine, if_exists="append", index=False, chunksize=200)

(22630, 4) ['id', 'klasifikasi', 'sub_klasifikasi', 'paragraphs']
215338


215338